In [0]:
from pyspark.sql import functions as F

conc = spark.table("workspace.enigh.bronce_concentrado")
gast = spark.table("workspace.enigh.bronce_gastos")

def num(c):
    return F.expr(f"try_cast(trim({c}) as double)")

# ---------- CONCENTRADO ----------
texto = ["folioviv", "foliohog", "ubica_geo", "upm"]
enteros = ["tam_loc", "est_socio", "est_dis", "clase_hog", "sexo_jefe", "edad_jefe", "educa_jefe",
           "tot_integ", "hombres", "mujeres", "mayores", "menores", "p12_64", "p65mas",
           "ocupados", "percep_ing", "perc_ocupa"]
meta = ["_fecha_ingesta", "_archivo_origen"]
decimales = [c for c in conc.columns if c not in texto + enteros + meta]

s_conc = (conc.select(
            F.lpad(F.trim("folioviv"), 10, "0").alias("folioviv"),
            F.trim("foliohog").alias("foliohog"),
            F.lpad(F.trim("ubica_geo"), 5, "0").alias("ubica_geo"),
            F.trim("upm").alias("upm"),
            *[num(c).cast("int").alias(c) for c in enteros],
            *[num(c).alias(c) for c in decimales])
          .withColumn("entidad", F.substring("ubica_geo", 1, 2))
          .dropDuplicates(["folioviv", "foliohog"]))

# ---------- GASTOS ----------
s_gast = (gast
    .withColumn("gasto_mon_tri", num("gasto_tri"))
    .withColumn("gasto_nm_tri", num("gas_nm_tri"))
    .withColumn("gasto_total_tri", F.coalesce(F.col("gasto_mon_tri"), F.lit(0.0)) + F.coalesce(F.col("gasto_nm_tri"), F.lit(0.0)))
    .filter(F.col("gasto_total_tri") > 0)
    .select(
        F.lpad(F.trim("folioviv"), 10, "0").alias("folioviv"),
        F.trim("foliohog").alias("foliohog"),
        F.trim("clave").alias("clave"),
        F.substring(F.trim("clave"), 1, 3).alias("grupo3"),
        F.trim("tipo_gasto").alias("tipo_gasto"),
        F.trim("forma_pag1").alias("forma_pag1"),
        F.trim("lugar_comp").alias("lugar_comp"),
        F.trim("frecuencia").alias("frecuencia"),
        num("cantidad").alias("cantidad"),
        "gasto_mon_tri", "gasto_nm_tri", "gasto_total_tri",
        num("factor").alias("factor")))

s_conc.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.enigh.silver_concentrado")
s_gast.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.enigh.silver_gastos")

# ---------- REPORTE ----------
sc = spark.table("workspace.enigh.silver_concentrado")
sg = spark.table("workspace.enigh.silver_gastos")
n_sc, n_sg = sc.count(), sg.count()
print("silver_concentrado:", n_sc, "filas")
print("silver_gastos:", n_sg, "filas | descartadas:", 5311497 - n_sg)

cols_rev = enteros + ["ing_cor", "entidad"]
nulos = sc.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols_rev]).collect()[0].asDict()
print("Nulos en concentrado (solo columnas con nulos):", {k: v for k, v in nulos.items() if v})

for c in ["sexo_jefe", "educa_jefe", "tam_loc", "est_socio", "clase_hog"]:
    print(c, sorted([r[0] for r in sc.select(c).distinct().collect() if r[0] is not None]))

silver_concentrado: 91414 filas
silver_gastos: 5305197 filas | descartadas: 6300
Nulos en concentrado (solo columnas con nulos): {}
sexo_jefe [1, 2]
educa_jefe [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
tam_loc [1, 2, 3, 4]
est_socio [1, 2, 3, 4]
clase_hog [1, 2, 3, 4, 5]
